# `pycensuskr` example: load and visualize total income tax in South Korea

This notebook is a self-guided navigatory material for `pycensuskr` package. As a twin package of the original package `tidycensuskr`, `pycensuskr` is aimed at offering the similar interface as that of `tidycensuskr` while observing Python programming practices. A simple example below will load and visualize the total income tax by basic local governments in South Korea. For getting basic knowledge base on South Korean geographic hierarchies, consult the [`tidycensuskr` vignettes](https://sigmafelix.r-universe.dev/tidycensuskr).

In [ ]:
from pycensuskr import CensusKR
from matplotlib import pyplot as plt
import geopandas as gpd

## Initialize CensusKR instance
Users should create an instance of `CensusKR()` to use all methods provided from it.

In [ ]:
# Create a CensusData instance
census = CensusKR()

## Load all data: `load_data()`
Users are expected to use `anycensus()` method described in the following section to load specific and fine-grained data most of the time. However, if one is interested to explore the entire dataset provided with this package, `load_data()` will serve for that purpose.

In [ ]:
# load specific year data
data_2020 = census.load_data(2020)
print(data_2020)

## Load tax data
Tax data are called with `anycensus()` method. `aggregator` argument will define the way of aggregating values stored in multiple rows per district, which usually occurs when there are subunits or data values collected at different hierarchy. One challenge arises here. That is tax statistics are recorded at basic local governments (BLG). BLGs are distinguished from _Si-Gun-Gus (SGG)_ in a way that each BLG has its fiscal autonomy, whereas some SGGs do not. This issue brings another step in cleaning boundary data. 

In [ ]:
# cleaned data with variable types
df_tax_2020 = census.anycensus(year = 2020, type = "tax", aggregator = "sum")

## Load district boundary and aggregating to BLGs
`load_districts()` with the required argument `year` loads district boundaries in polygons. The district here stands for SGG, which means one need to aggregate nonautonomous SGG into BLGs.

In [ ]:
# load district boundaries for a specific year
districts_2020 = census.load_districts(2020)


`adm2_code` field stores the management code for SGGs. The codes are organized systematically with the first two digits for provinces (_Si-Do_) and the remaining three for SGGs. In detail, the last three digits can be further dissected to two and one digits for BLGs and nonautonomous districts (if exists), respectively. The code block below utilizes such characteristics to add a new column named `adm2_re` to dissolve polygons.

In [ ]:
districts_2020["adm2_re"] = districts_2020["adm2_code"].astype(str).str.slice(0,4)

# aggregate geometries by adm2_re
districts_2020 = districts_2020.dissolve(by="adm2_re", as_index=False)
districts_2020["adm2_code"] = districts_2020["adm2_re"] + "0"
districts_2020["adm2_code"] = districts_2020["adm2_code"].astype(int)

## Merge and visualize tax data
Finally, the merged boundary is ready to be merged with the tax data. Below demonstrates the total labor income tax in million Korean Won (inflation unadjusted) by BLGs.

In [ ]:
districts_tax_2020 = districts_2020.merge(df_tax_2020, on="adm2_code")
print(districts_tax_2020)

districts_tax_2020.plot("income_labor_mil")
plt.show()
